# Phase 1: Hillstrom Uplift Modeling Walkthrough

This notebook ties together the Phase 1 pipeline:

1. Prep the MineThatData **Hillstrom** email RCT (Mens E-Mail vs No E-Mail)
2. Fit a **T-learner** and **CausalForestDML**
3. Evaluate with **Qini** and **uplift-at-k** (not accuracy / AUC)

Goal: prove correct uplift methodology on a real randomized experiment before Phase 2 (synthetic credit-limit uplift).

## Why accuracy and AUC are the wrong metrics here

If you come from classification, the instinct is: *predict visit, then check accuracy or ROC-AUC*. That answers a different question.

| Framing | Question answered | Typical metric |
|---|---|---|
| Classification | "Who will visit?" | Accuracy, AUC |
| Uplift / CATE | "Whose visit probability **changes because of** the email?" | Qini, uplift-at-k |

**Why classification metrics fail for targeting:**

- **Sure things** already visit without the email. A good classifier ranks them highly. Targeting them wastes budget — treatment effect ≈ 0.
- **Lost causes** never visit either way. Classifier ranks them low (good for classification), but uplift is also ≈ 0.
- **Persuadables** only visit *because* of the email. That is who we want. Their baseline visit rate may be middling, so a visit-classifier often deprioritizes them.
- **Do-not-disturb** may even respond *worse* to treatment (negative CATE).

Uplift models estimate **CATE(x) = P(Y=1 | X=x, T=1) − P(Y=1 | X=x, T=0)**. Evaluation must score how well we rank people by that incremental effect — hence Qini curves and uplift-at-k on a held-out slice of the RCT.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

PROJECT_ROOT = C:\Users\tjayb\OneDrive\Desktop\uplift-modeling-credit


## 1. Data prep

Hillstrom is a **genuine randomized experiment**: `segment` was randomly assigned, so we do **not** need selection-bias / propensity correction for identification of ATE/CATE here. We still estimate propensity inside CausalForestDML as a nuisance model (DML machinery), but the experimental design is what makes the causal claim clean.

Binary treatment for this phase:
- **Treatment (T=1):** Mens E-Mail
- **Control (T=0):** No E-Mail
- Drop Womens E-Mail

Primary outcome: **`visit`**. `spend` / `conversion` are kept in processed parquet for later, not modeled now.

In [2]:
from data_prep import run_data_prep, feature_columns

train_df, test_df = run_data_prep()
print("Feature columns:", feature_columns(train_df))
train_df.head()

Loaded raw: 64,000 rows, columns=['customer_id', 'recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'visit', 'segment', 'conversion', 'spend']
After binary filter (Mens vs No E-Mail): 42,613 rows | treatment rate=0.5000

=== Randomization / Sanity Check (TRAIN) ===
Train n = 34,090
Treatment share = 0.5000 (expect ~0.50 for binary Mens vs No E-Mail)
mean(visit) | treated = 0.1828 | control = 0.1062 | ATE (naive) = 0.0766

Covariate balance (randomization check - means should be similar):
  recency     treated=5.7748  control=5.7620  diff=0.0127
  history     treated=243.8256  control=241.0399  diff=2.7857
=== End Sanity Check ===

Wrote C:\Users\tjayb\OneDrive\Desktop\uplift-modeling-credit\data\processed\train.parquet (34,090 rows)
Wrote C:\Users\tjayb\OneDrive\Desktop\uplift-modeling-credit\data\processed\test.parquet (8,523 rows)
Feature columns: ['recency', 'history', 'mens', 'womens', 'newbie', 'history_segment_1) $0 - $100', 'history_segme

,customer_id,recency,history,mens,womens,newbie,visit,conversion,spend,treatment,...,history_segment_4) $350 - $500,history_segment_5) $500 - $750,"history_segment_6) $750 - $1,000","history_segment_7) $1,000 +",zip_code_Rural,zip_code_Surburban,zip_code_Urban,channel_Multichannel,channel_Phone,channel_Web
0,9185,6,475.04,0,1,0,1,0,0.0,0,...,1,0,0,0,0,1,0,1,0,0
1,40738,4,50.37,0,1,1,0,0,0.0,1,...,0,0,0,0,0,1,0,0,0,1
2,26490,2,117.41,0,1,1,0,0,0.0,0,...,0,0,0,0,0,1,0,0,1,0
3,20581,4,151.95,1,0,0,0,0,0.0,1,...,0,0,0,0,1,0,0,0,1,0
4,27931,10,50.49,0,1,0,0,0,0.0,0,...,0,0,0,0,0,1,0,0,0,1


## 2. T-learner

Two separate outcome models:

- μ₁(x) = P(visit | X=x) trained on **treated** only
- μ₀(x) = P(visit | X=x) trained on **control** only

Then **ĈATE(x) = μ₁(x) − μ₀(x)** on every test row.

Default base learner: `GradientBoostingClassifier` (pass `model_name="logistic"` for a linear baseline).

In [3]:
from models.t_learner import run_t_learner

t_preds = run_t_learner(model_name="gbm")
t_preds.head()

T-learner (gbm) predictions saved to C:\Users\tjayb\OneDrive\Desktop\uplift-modeling-credit\outputs\t_learner_predictions.csv (8,523 rows)
  predicted_cate mean=0.0763 std=0.0404


,customer_id,true_treatment,true_outcome,predicted_cate,p_visit_treated,p_visit_control
0,26086,0,0,0.052547,0.106749,0.054202
1,51760,0,0,0.030869,0.142560,0.111691
2,62650,0,0,0.051747,0.073479,0.021732
3,58597,0,0,0.062676,0.110061,0.047384
4,27252,0,0,0.062837,0.109293,0.046456


## 3. Causal forest (CausalForestDML)

We use **`econml.dml.CausalForestDML`** rather than `LinearDML` because the point of this phase is **heterogeneous** effects — a forest final stage learns flexible CATE(x), while DML residualizes Y and T so the CATE stage is more robust to nuisance misspecification.

If econml fails, this module raises (no silent fallback to causalml).

In [4]:
from models.causal_forest import run_causal_forest

cf_preds = run_causal_forest()
cf_preds.head()

CausalForestDML predictions saved to C:\Users\tjayb\OneDrive\Desktop\uplift-modeling-credit\outputs\causal_forest_predictions.csv (8,523 rows)
  predicted_cate mean=0.0752 std=0.0093


,customer_id,true_treatment,true_outcome,predicted_cate
0,26086,0,0,0.068185
1,51760,0,0,0.062098
2,62650,0,0,0.070608
3,58597,0,0,0.067730
4,27252,0,0,0.066166


## 4. Uplift metrics (from definition)

### Qini curve / coefficient
Rank everyone by predicted CATE (high → low). Walking down that list, the Qini curve tracks **incremental outcomes** attributable to treating the ranked slice versus what random targeting would achieve. The **Qini coefficient** is the area between the model curve and the random-targeting diagonal — larger is better.

### Uplift-at-k
Among the top *k*% of customers by predicted CATE:

`uplift@k = mean(visit | treated, top-k) − mean(visit | control, top-k)`

This approximates the treatment effect in the segment you would actually mail if budget only covers *k*% of the list — the practical targeting question.

In [5]:
from evaluation.uplift_metrics import run_evaluation

summary = run_evaluation()
summary

Saved Qini plot to C:\Users\tjayb\OneDrive\Desktop\uplift-modeling-credit\outputs\qini_curve.png

=== Uplift Metrics Summary ===
          model  qini_coefficient  uplift_at_10pct  uplift_at_30pct  uplift_at_50pct
      T-learner            7.7372           0.1159           0.0942           0.0803
CausalForestDML            6.2569           0.1533           0.0858           0.0810
=== End Summary ===



,model,qini_coefficient,uplift_at_10pct,uplift_at_30pct,uplift_at_50pct
0,T-learner,7.737243,0.115931,0.094216,0.080278
1,CausalForestDML,6.256883,0.153302,0.085787,0.080998


## 5. Who are the "Persuadables" in this data?

Inspect the top decile of test customers by **T-learner** predicted CATE and compare feature means to the rest of the test set.

In [6]:
# Merge CATE scores back to test features for profile analysis
profile = test_df.merge(
    t_preds[["customer_id", "predicted_cate"]],
    on="customer_id",
    how="inner",
)
cutoff = profile["predicted_cate"].quantile(0.9)
top = profile[profile["predicted_cate"] >= cutoff]
rest = profile[profile["predicted_cate"] < cutoff]

profile_cols = ["recency", "history", "mens", "womens", "newbie", "visit", "treatment"]
compare = pd.DataFrame(
    {
        "top_decile_mean": top[profile_cols].mean(),
        "rest_mean": rest[profile_cols].mean(),
    }
)
compare["diff"] = compare["top_decile_mean"] - compare["rest_mean"]
print(f"Top-decile n={len(top):,} | CATE cutoff={cutoff:.4f}")
print(f"Top-decile mean predicted CATE={top['predicted_cate'].mean():.4f}")
print(f"Rest mean predicted CATE={rest['predicted_cate'].mean():.4f}")
compare

Top-decile n=853 | CATE cutoff=0.1206
Top-decile mean predicted CATE=0.1555
Rest mean predicted CATE=0.0675


,top_decile_mean,rest_mean,diff
recency,4.283705,5.896089,-1.612384
history,459.003154,215.161426,243.841727
mens,0.822978,0.507953,0.315025
womens,0.776084,0.537679,0.238405
newbie,0.558030,0.490743,0.067287
visit,0.179367,0.140548,0.038819
treatment,0.466589,0.503781,-0.037192


In [7]:
# Plain-English summary from metrics + top-decile feature profile
t_row = summary[summary["model"] == "T-learner"].iloc[0]
cf_row = summary[summary["model"] == "CausalForestDML"].iloc[0]

better = (
    "T-learner"
    if t_row["qini_coefficient"] >= cf_row["qini_coefficient"]
    else "CausalForestDML"
)
delta_qini = abs(float(t_row["qini_coefficient"]) - float(cf_row["qini_coefficient"]))

print("=== Plain-English summary ===\n")
print(
    f"{better} wins on overall ranking quality (Qini coefficient), "
    f"by {delta_qini:.4f} absolute points "
    f"(T-learner Qini={float(t_row['qini_coefficient']):.4f} vs "
    f"CausalForestDML Qini={float(cf_row['qini_coefficient']):.4f}). "
    f"At a tight budget (top 10%), CausalForestDML uplift@10%="
    f"{float(cf_row['uplift_at_10pct']):.4f} vs T-learner "
    f"{float(t_row['uplift_at_10pct']):.4f}; at 30%/50% the gap narrows "
    f"(T-learner {float(t_row['uplift_at_30pct']):.4f}/{float(t_row['uplift_at_50pct']):.4f}, "
    f"CF {float(cf_row['uplift_at_30pct']):.4f}/{float(cf_row['uplift_at_50pct']):.4f})."
)
print()

# Describe persuadables from the actual top-decile profile
d = compare["diff"]
bits = []
if d["mens"] > 0.02:
    bits.append("more likely to have bought Mens merchandise before")
elif d["mens"] < -0.02:
    bits.append("less likely to have bought Mens merchandise before")
if d["womens"] > 0.02:
    bits.append("more likely to have bought Womens merchandise before")
elif d["womens"] < -0.02:
    bits.append("less likely to have bought Womens merchandise before")
if d["newbie"] > 0.02:
    bits.append("more often newbies")
elif d["newbie"] < -0.02:
    bits.append("less often newbies")
if d["recency"] > 0.2:
    bits.append(f"longer time since last purchase (recency +{d['recency']:.1f} months)")
elif d["recency"] < -0.2:
    bits.append(f"more recent purchasers (recency {d['recency']:.1f} months)")
if d["history"] > 10:
    bits.append(f"higher past spend (history +${d['history']:.0f})")
elif d["history"] < -10:
    bits.append(f"lower past spend (history ${d['history']:.0f})")

profile_clause = "; ".join(bits) if bits else "similar averages on the main covariates"
print(
    f"In this fitted T-learner, the Persuadables segment (top-decile predicted CATE, "
    f"n={len(top):,}, mean CATE={top['predicted_cate'].mean():.4f}) looks like customers who are "
    f"{profile_clause}, relative to everyone else in the test set. "
    "That pattern is what you'd expect for a Mens E-Mail treatment: incremental visits "
    "concentrate where the creative matches prior category affinity and where baseline "
    "engagement isn't already maxed out (sure things). Phase 2 will ask whether a "
    "credit-limit offer shows an analogous 'who benefits' slice under synthetic RCT data."
)

=== Plain-English summary ===

T-learner wins on overall ranking quality (Qini coefficient), by 1.4804 absolute points (T-learner Qini=7.7372 vs CausalForestDML Qini=6.2569). At a tight budget (top 10%), CausalForestDML uplift@10%=0.1533 vs T-learner 0.1159; at 30%/50% the gap narrows (T-learner 0.0942/0.0803, CF 0.0858/0.0810).

In this fitted T-learner, the Persuadables segment (top-decile predicted CATE, n=853, mean CATE=0.1555) looks like customers who are more likely to have bought Mens merchandise before; more likely to have bought Womens merchandise before; more often newbies; more recent purchasers (recency -1.6 months); higher past spend (history +$244), relative to everyone else in the test set. That pattern is what you'd expect for a Mens E-Mail treatment: incremental visits concentrate where the creative matches prior category affinity and where baseline engagement isn't already maxed out (sure things). Phase 2 will ask whether a credit-limit offer shows an analogous 'who b

## 6. Who is the model finding? (segment profiles)

The cell below loads **saved** predictions only (no retraining) via `src/evaluation/segment_profile.py`. Categoricals (`history_segment`, `zip_code`, `channel`) are reconstructed from one-hot columns inside that script.

In [ ]:
from evaluation.segment_profile import run_segment_profile

seg_summary, seg_overlap, seg_narrative = run_segment_profile()
seg_summary

### Plain-English takeaway (Persuadables & model agreement)

**Persuadables** (top 10% by predicted CATE) in this experiment look like more recent, higher-spend customers who already buy in both Mens and Womens categories (mens ≈ 82–87%, womens ≈ 78–84%), lean slightly newbie, and concentrate in mid history bands (~$200–$350), Urban zips, and Web. That is not “sure things only”: inside that top decile the **observed** visit gap is real — about +12 pp (T-learner) to +15 pp (CausalForestDML) treated vs control — so the models are pointing at people who actually respond more to Mens E-Mail in the holdout RCT.

**Low-uplift** (bottom 10%) is milder, not a classic Sleeping Dogs segment: both models’ bottom-decile mean CATE stays non-negative (T-learner still has ~14% individual negative scores; CausalForestDML has none). Those customers look lower-spend / lower history-segment, more Suburban, and — especially under CausalForestDML — more stale on recency with a Mens-heavy, Womens-light mix. Observed uplift in the bottom decile is still positive (~+10 pp) but smaller than the top, so “low uplift” here means *relatively* less incremental lift, not harm.

**Model disagreement on membership is real:** only **58%** of each model’s top-decile customer_ids overlap (Jaccard ≈ 0.41). Feature *tilts* agree more than the exact mailing lists — shared direction (recent, higher history, category-engaged, Urban/Web) is usable; treating either top-10% ID list as ground truth is not.

**Implication for Phase 2 synthetic design:** build heterogeneity around features that moved together here — recency, past spend / history band, and category affinity — as a loose template for “who benefits,” not as a claim that credit customers behave like apparel email recipients. Prefer effects that remain stable across estimators, given how much the two models diverge on who sits in the top decile.